# Baseline Analysis

Goal: Understand the CICDS data and what the strengths and weaknesses of the data are (imbalances, etc)
- Data Preprocessing and formatting
- Removing undersized minority classes (ex. 1 attack instance) and applying SMOTE
- Run a class analyis on the data for sizes of classes
- Train the XGBoost version on a subset of the data (10% of original data for now)

In [1]:
import resource

# Limit Python to 20GB - prevents system freeze
resource.setrlimit(resource.RLIMIT_AS, (20 * 1024 * 1024 * 1024, -1))

In [2]:
# Cell 0: Imports
# ==============================================================================
import xgboost as xgb
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.preprocessing import LabelEncoder
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score
from pathlib import Path
import pickle
import time
import gc
import warnings

warnings.filterwarnings('ignore')
sns.set_style("whitegrid")
plt.rcParams['figure.figsize'] = (12, 6)

print("="*70)
print("CICIDS2017 ANALYSIS - MULTI-CELL JUPYTER NOTEBOOK")
print("="*70)

# SET MEMORY LIMITS (PREVENT SYSTEM FREEZE!)
import resource
resource.setrlimit(resource.RLIMIT_AS, (20 * 1024 * 1024 * 1024, -1))
print("✓ Memory limit set to 20GB")

# Configuration
SAMPLE_FRACTION = 0.10  # 10% sampling = safe, fast
MIN_SAMPLES_PER_CLASS = 10
RANDOM_STATE = 42

# Setup paths
DATA_PATH = Path("./data")
RESULTS_PATH = Path("./results")
MODELS_PATH = Path("./models")

RESULTS_PATH.mkdir(exist_ok=True)
MODELS_PATH.mkdir(exist_ok=True)

print(f"\n✓ Paths configured")
print(f"  Data: {DATA_PATH.absolute()}")
print(f"  Results: {RESULTS_PATH.absolute()}")
print(f"  Models: {MODELS_PATH.absolute()}")

CICIDS2017 ANALYSIS - MULTI-CELL JUPYTER NOTEBOOK
✓ Memory limit set to 20GB

✓ Paths configured
  Data: /home/cpre560/Desktop/IDS-ML/notebooks/data
  Results: /home/cpre560/Desktop/IDS-ML/notebooks/results
  Models: /home/cpre560/Desktop/IDS-ML/notebooks/models


In [7]:
# Cell 1: Loading dataset
# ==============================================================================
print("\n" + "="*70)
print("LOADING CICIDS2017 DATA")
print("="*70)

# Find CSV files
csv_files = sorted(DATA_PATH.glob("*.csv"))
print(f"Found {len(csv_files)} CSV files:")

for csv_file in csv_files:
    file_size = csv_file.stat().st_size / (1024*1024)
    print(f"  ✓ {csv_file.name} ({file_size:.1f} MB)")

# Map files by keyword
def find_file_by_keyword(keyword, files):
    for f in files:
        if keyword.lower() in f.name.lower():
            return f
    return None

days_keywords = {
    "Monday": "Monday",
    "Tuesday": "Tuesday",
    "Wednesday": "Wednesday",
    "Thursday_Morning": "Morning",
    "Thursday_Afternoon": "Infiltr",
    "Friday_Morning": "Friday",
    "Friday_PortScan": "PortScan",
    "Friday_DDoS": "DDoS"
}

files_to_load = {k: v for k, v in
                 {day: find_file_by_keyword(kw, csv_files)
                  for day, kw in days_keywords.items()}.items()
                 if v is not None}

# Load files
print(f"\nLoading {len(files_to_load)} files...")
start_time = time.time()
dfs = {}
total_records = 0

for day_label, filepath in files_to_load.items():
    print(f"  {day_label:20s}...", end=" ", flush=True)
    df = pd.read_csv(filepath)
    dfs[day_label] = df
    total_records += len(df)
    print(f"✓ ({len(df):,} records)")
    gc.collect()  # Free memory after each file

# Combine files
df_combined = pd.concat(dfs.values(), ignore_index=True)
load_time = time.time() - start_time

print(f"\n✓ Loading complete ({load_time:.1f}s)")
print(f"  Combined shape: {df_combined.shape}")
print(f"  Total records: {total_records:,}")



LOADING CICIDS2017 DATA
Found 0 CSV files:

Loading 0 files...


ValueError: No objects to concatenate

In [ ]:
# Cell 2: Data Cleanng and Preprocessing
# ==============================================================================
print("\n" + "="*70)
print("DATA CLEANING & PREPROCESSING")
print("="*70)

# Auto-detect label column
label_col = df_combined.columns[-1]
print(f"\nLabel column: '{label_col}'")
print(f"Unique classes before cleaning: {df_combined[label_col].nunique()}")

# Remove NaN values
df_cleaned = df_combined.dropna()
print(f"After removing NaN: {len(df_cleaned):,} records")

# Sample dataset (memory optimization)
print(f"\nSampling {SAMPLE_FRACTION*100:.0f}% of data...")
df_sampled = df_cleaned.groupby(label_col, group_keys=False).apply(
    lambda x: x.sample(frac=SAMPLE_FRACTION, random_state=RANDOM_STATE)
)
df_cleaned = df_sampled
print(f"Sampled size: {len(df_cleaned):,} records")
gc.collect()

# Remove extremely rare classes
print(f"\nFiltering classes with < {MIN_SAMPLES_PER_CLASS} samples...")
class_counts_before = df_cleaned[label_col].value_counts()
classes_to_keep = class_counts_before[class_counts_before >= MIN_SAMPLES_PER_CLASS].index
classes_removed = class_counts_before[class_counts_before < MIN_SAMPLES_PER_CLASS].index

if len(classes_removed) > 0:
    print(f"Removing {len(classes_removed)} rare classes:")
    for cls in classes_removed:
        print(f"  - {cls}: {class_counts_before[cls]} sample(s)")
    df_cleaned = df_cleaned[df_cleaned[label_col].isin(classes_to_keep)]
    print(f"Records after filtering: {len(df_cleaned):,}")

# ============================================================================
# CLASS DISTRIBUTION ANALYSIS
# ============================================================================

print("\n" + "="*70)
print("CLASS DISTRIBUTION ANALYSIS")
print("="*70)

class_counts = df_cleaned[label_col].value_counts()
class_percentages = (class_counts / len(df_cleaned) * 100).round(4)

print(f"\n{'Attack Type':<30} {'Count':>10} {'% of Dataset':>15} {'vs Majority':>15}")
print("-" * 70)
for cls, count in class_counts.items():
    pct = class_percentages[cls]
    vs_maj = (count / class_counts.values[0] * 100)
    print(f"{cls:<30} {count:>10,} {pct:>14.2f}% {vs_maj:>14.6f}%")

# Save class distribution
imbalance_df = pd.DataFrame({
    'Attack Type': class_counts.index,
    'Count': class_counts.values,
    '% of Dataset': class_percentages.values,
})
imbalance_df.to_csv(RESULTS_PATH / "class_distribution.csv", index=False)
print(f"\n✓ Class distribution saved")

# Visualize class distribution
fig, axes = plt.subplots(1, 2, figsize=(15, 5))

# Bar chart (log scale)
axes[0].barh(range(len(class_counts)), class_counts.values, color='steelblue')
axes[0].set_yticks(range(len(class_counts)))
axes[0].set_yticklabels(class_counts.index, fontsize=9)
axes[0].set_xlabel('Number of Instances (Log Scale)')
axes[0].set_title('CICIDS2017 Class Distribution (After Sampling & Filtering)')
axes[0].set_xscale('log')
axes[0].grid(axis='x', alpha=0.3)

# Pie chart
top_n = min(6, len(class_percentages))
top_classes = class_percentages.nlargest(top_n)
other_count = class_percentages.iloc[top_n:].sum() if len(class_percentages) > top_n else 0
pie_data = list(top_classes.values) + ([other_count] if other_count > 0 else [])
pie_labels = list(top_classes.index) + (['Others'] if other_count > 0 else [])

axes[1].pie(pie_data, labels=pie_labels, autopct='%1.2f%%', startangle=90)
axes[1].set_title('Attack Type Distribution')

plt.tight_layout()
plt.savefig(RESULTS_PATH / 'class_imbalance.png', dpi=300, bbox_inches='tight')
plt.show()

print("✓ Distribution chart saved")


In [ ]:
# Cell 4: Feature Prep and Extraction
# ==============================================================================
print("\n" + "="*70)
print("FEATURE PREPARATION & ENCODING")
print("="*70)

# Extract features and target
feature_cols = [col for col in df_cleaned.columns if col != label_col]
X = df_cleaned[feature_cols].copy()
y = df_cleaned[label_col].copy()

print(f"\nFeatures: {X.shape}")
print(f"Target: {y.shape}")

# Identify categorical columns
categorical_cols = X.select_dtypes(include=['object']).columns.tolist()
print(f"\nCategorical features: {len(categorical_cols)}")
if categorical_cols:
    print(f"  {categorical_cols[:5]}{'...' if len(categorical_cols) > 5 else ''}")

# Encode categorical features
print(f"\nEncoding categorical features...")
label_encoders = {}

for col in categorical_cols:
    le = LabelEncoder()
    X[col] = le.fit_transform(X[col].astype(str))
    label_encoders[col] = le

print(f"✓ {len(categorical_cols)} categorical features encoded")

# Encode target variable
print(f"\nEncoding target variable...")
le_target = LabelEncoder()
y_encoded = le_target.fit_transform(y)

print(f"✓ Classes ({len(le_target.classes_)}): {list(le_target.classes_)}")

# Handle inf/NaN values
print(f"\nHandling inf/NaN values...")
X = X.replace([np.inf, -np.inf], np.nan)
X = X.fillna(X.median())

print(f"✓ Data preparation complete")
print(f"  Features shape: {X.shape}")
print(f"  Target classes: {len(le_target.classes_)}")

gc.collect()


In [ ]:
# CELL 6: TRAIN-TEST SPLIT (CORRECTED)
# ==============================================================================

print("\n" + "="*70)
print("TRAIN-TEST SPLIT")
print("="*70)

# Check if stratified split is possible (use numpy bincount, not pandas value_counts)
class_counts_array = np.bincount(y_encoded)
min_class_count = class_counts_array.min()
print(f"Minimum samples in any class: {min_class_count}")

if min_class_count < 2:
    print("⚠ Using non-stratified split (min samples < 2)")
    X_train, X_test, y_train, y_test = train_test_split(
        X, y_encoded, test_size=0.2, random_state=RANDOM_STATE, stratify=None
    )
else:
    print("✓ Using stratified split (maintains class distribution)")
    X_train, X_test, y_train, y_test = train_test_split(
        X, y_encoded, test_size=0.2, random_state=RANDOM_STATE, stratify=y_encoded
    )

print(f"\nTrain set: {X_train.shape[0]:,} records")
print(f"Test set: {X_test.shape[0]:,} records")

# Show class distribution in train set (use numpy bincount instead of pandas)
print(f"\nClass distribution in training set:")
train_counts = np.bincount(y_train)
for i, class_name in enumerate(le_target.classes_):
    if i < len(train_counts):
        count = train_counts[i]
        pct = count / len(y_train) * 100
        print(f"  {class_name:30s}: {count:>8,} ({pct:>6.2f}%)")

gc.collect()


In [ ]:
# CELL 7: TRAIN XGBOOST MODEL
# ==============================================================================

print("\n" + "="*70)
print("TRAINING XGBOOST CLASSIFIER")
print("="*70)

print(f"XGBoost version: {xgb.__version__}")
print(f"Training samples: {X_train_balanced.shape[0]:,}")
print(f"Test samples: {X_test.shape[0]:,}")

print(f"\nXGBoost Configuration:")
print(f"  n_estimators: 100")
print(f"  max_depth: 6")
print(f"  learning_rate: 0.1")
print(f"  tree_method: hist (CPU optimized)")

# Create and train model
model = xgb.XGBClassifier(
    n_estimators=100,
    max_depth=6,
    learning_rate=0.1,
    tree_method='hist',
    objective='multi:softmax',
    num_class=len(le_target.classes_),
    random_state=RANDOM_STATE,
    n_jobs=-1,
    eval_metric='mlogloss',
    verbosity=0,
)

print(f"\nTraining...")
start_train = time.time()

try:
    model.fit(X_train_balanced, y_train_balanced, verbose=False)
    train_time = time.time() - start_train
    print(f"✓ Training complete ({train_time:.2f}s)")
except Exception as e:
    print(f"✗ Training failed: {str(e)[:100]}")
    raise

gc.collect()

In [ ]:
# CELL 8: FEATURE IMPORTANCE ANALYSIS
# ==============================================================================

print("\n" + "="*70)
print("FEATURE IMPORTANCE ANALYSIS")
print("="*70)

# Calculate feature importance
feature_importance = pd.DataFrame({
    'Feature': X_train.columns,
    'Importance': model.feature_importances_
}).sort_values('Importance', ascending=False)

feature_importance['Cumulative %'] = (
    feature_importance['Importance'].cumsum() /
    feature_importance['Importance'].sum() * 100
).round(2)

print("\nTop 20 Most Important Features:")
print("-" * 70)
for idx, row in feature_importance.head(20).iterrows():
    print(f"{row['Feature']:40s} {row['Importance']:.6f}")

# Calculate thresholds
features_80 = (feature_importance['Cumulative %'] <= 80).sum()
features_95 = (feature_importance['Cumulative %'] <= 95).sum()

print(f"\n✓ Features for 80% importance: {features_80} out of {len(feature_importance)}")
print(f"✓ Features for 95% importance: {features_95} out of {len(feature_importance)}")

# Save feature importance
feature_importance.to_csv(RESULTS_PATH / 'feature_importance.csv', index=False)
print(f"✓ Saved: feature_importance.csv")

# Visualize feature importance
fig, ax = plt.subplots(figsize=(12, 8))
top_features = feature_importance.head(20)
ax.barh(range(len(top_features)), top_features['Importance'].values, color='steelblue')
ax.set_yticks(range(len(top_features)))
ax.set_yticklabels(top_features['Feature'].values, fontsize=9)
ax.set_xlabel('Importance Score')
ax.set_title('Top 20 Most Important Features')
ax.invert_yaxis()
ax.grid(axis='x', alpha=0.3)

plt.tight_layout()
plt.savefig(RESULTS_PATH / 'feature_importance.png', dpi=300, bbox_inches='tight')
plt.show()

print("✓ Saved: feature_importance.png")

gc.collect()

In [ ]:
# CELL 9: MODEL EVALUATION
# ==============================================================================
print("\n" + "="*70)
print("MODEL EVALUATION")
print("="*70)

# Make predictions
print(f"Making predictions on {len(X_test):,} test samples...")
start_pred = time.time()
y_pred = model.predict(X_test)
y_pred_proba = model.predict_proba(X_test)
pred_time = time.time() - start_pred

print(f"✓ Prediction complete ({pred_time:.3f}s)")

# Calculate accuracy
accuracy = accuracy_score(y_test, y_pred)
print(f"\nOverall Accuracy: {accuracy*100:.2f}%")
print(f"Inference Speed: {len(X_test)/pred_time:.0f} samples/second")

# Classification report
print("\n" + "="*70)
print("CLASSIFICATION REPORT")
print("="*70)
print(classification_report(y_test, y_pred, target_names=le_target.classes_))

# Save classification report
report_dict = classification_report(y_test, y_pred, target_names=le_target.classes_,
                                    output_dict=True)
report_df = pd.DataFrame(report_dict).transpose()
report_df.to_csv(RESULTS_PATH / 'classification_report.csv')
print("✓ Saved: classification_report.csv")

# Confusion matrix
cm = confusion_matrix(y_test, y_pred)
print(f"\nConfusion Matrix shape: {cm.shape}")

# Visualize confusion matrix
fig, ax = plt.subplots(figsize=(12, 10))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=le_target.classes_,
            yticklabels=le_target.classes_,
            ax=ax, cbar_kws={'label': 'Count'})
ax.set_title('Confusion Matrix - Baseline XGBoost')
ax.set_ylabel('True Label')
ax.set_xlabel('Predicted Label')
plt.xticks(rotation=45, ha='right')
plt.tight_layout()
plt.savefig(RESULTS_PATH / 'confusion_matrix.png', dpi=300, bbox_inches='tight')
plt.show()

print("✓ Saved: confusion_matrix.png")

gc.collect()


In [ ]:
# CELL 10: SAVE RESULTS & SUMMARY
# ==============================================================================

print("\n" + "="*70)
print("SAVING RESULTS & SUMMARY")
print("="*70)

# Create metrics summary
metrics_summary = {
    'Original Dataset Size': f"{total_records:,}",
    'Sample Fraction': f"{SAMPLE_FRACTION*100:.0f}%",
    'Final Dataset Size': f"{len(df_cleaned):,}",
    'Classes Analyzed': len(le_target.classes_),
    'Features': len(feature_cols),
    'Training Samples (Balanced)': f"{X_train_balanced.shape[0]:,}",
    'Test Samples': f"{len(X_test):,}",
    'Train Time (s)': f"{train_time:.2f}",
    'Inference Speed (samples/s)': f"{len(X_test)/pred_time:.0f}",
    'Overall Accuracy': f"{accuracy*100:.2f}%",
    'Features for 80% Importance': features_80,
    'Features for 95% Importance': features_95,
}

print("\n" + "="*70)
print("BASELINE METRICS SUMMARY")
print("="*70)
for key, value in metrics_summary.items():
    print(f"{key:.<45s} {value}")

# Save summary
summary_df = pd.DataFrame(list(metrics_summary.items()), columns=['Metric', 'Value'])
summary_df.to_csv(RESULTS_PATH / 'metrics_summary.csv', index=False)
print(f"\n✓ Saved: metrics_summary.csv")

# Save model
model.save_model(str(MODELS_PATH / 'baseline_xgboost.json'))
print(f"✓ Saved: baseline_xgboost.json")

# Save preprocessing info
preprocessing_info = {
    'categorical_encoders': label_encoders,
    'target_encoder': le_target,
    'feature_columns': feature_cols,
    'categorical_columns': categorical_cols,
}

with open(MODELS_PATH / 'preprocessing_info.pkl', 'wb') as f:
    pickle.dump(preprocessing_info, f)

print(f"✓ Saved: preprocessing_info.pkl")


In [ ]:
# CELL 11: FINAL SUMMARY
# ==============================================================================

print("\n" + "="*70)
print("ANALYSIS COMPLETE!")
print("="*70)

print(f"\n📊 Results Directory: {RESULTS_PATH.absolute()}")
print(f"   Files:")
print(f"   - metrics_summary.csv")
print(f"   - class_distribution.csv")
print(f"   - feature_importance.csv")
print(f"   - classification_report.csv")
print(f"   - class_imbalance.png")
print(f"   - feature_importance.png")
print(f"   - confusion_matrix.png")

print(f"\n💾 Models Directory: {MODELS_PATH.absolute()}")
print(f"   Files:")
print(f"   - baseline_xgboost.json")
print(f"   - preprocessing_info.pkl")

print(f"\n🎯 Key Findings:")
print(f"   - Dataset: {len(df_cleaned):,} records ({SAMPLE_FRACTION*100:.0f}% of original)")
print(f"   - Classes: {len(le_target.classes_)}")
print(f"   - Features: {len(feature_cols)}")
print(f"   - Accuracy: {accuracy*100:.2f}%")
print(f"   - Top Feature: {feature_importance.iloc[0]['Feature']}")

print(f"\n✓ System remained responsive (memory limit: 20GB)")
print(f"✓ Ready for next iteration!")

print("\n" + "="*70)